# Task 2 — ML Model Training (XGBoost)
Training a gradient boosted tree classifier on UNSW-NB15 features.
All experiments tracked in MLflow. Model must beat rule baseline F1 by ≥15%.

In [17]:
import numpy as np
import json
import os
import joblib
import mlflow
import mlflow.sklearn
from pathlib import Path
from xgboost import XGBClassifier
from sklearn.metrics import (f1_score, precision_score,
    recall_score, confusion_matrix, classification_report)

print("All imports successful")

All imports successful


In [18]:
DATA_DIR = "../data/processed"

# ── Step 1: Load your teammate's data ────────────────────────
# These are the ONLY files you need from them. Nothing else.
X_train = np.load(f"{DATA_DIR}/X_train.npy")
X_test  = np.load(f"{DATA_DIR}/X_test.npy")
y_train = np.load(f"{DATA_DIR}/y_train.npy")
y_test  = np.load(f"{DATA_DIR}/y_test.npy")

with open(f"{DATA_DIR}/feature_names.json") as f:
    FEATURE_COLS = json.load(f)

print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}   y_test:  {y_test.shape}")
print(f"Train attack rate: {y_train.mean()*100:.1f}%")
print(f"Test attack rate:  {y_test.mean()*100:.1f}%")

X_train: (175341, 37)  y_train: (175341,)
X_test:  (82332, 37)   y_test:  (82332,)
Train attack rate: 68.1%
Test attack rate:  55.1%


In [19]:
# ── Step 2: Handle class imbalance ───────────────────────────
# UNSW-NB15 has more normal flows than attacks.
# scale_pos_weight tells XGBoost: "pay more attention to attacks"
# because they are the minority class we care most about catching.

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / pos
print(f"Normal flows: {neg:,}")
print(f"Attack flows: {pos:,}")
print(f"scale_pos_weight = {spw:.2f}")
print("(This tells XGBoost to pay more attention to the minority attack class)")

Normal flows: 56,000
Attack flows: 119,341
scale_pos_weight = 0.47
(This tells XGBoost to pay more attention to the minority attack class)


In [20]:
# ── Step 3: Connect to MLflow ─────────────────────────────────
# MLflow must be running: docker-compose up -d mlflow

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("secopsai-detection")
print("Connected to MLflow at localhost:5000")
print("Open localhost:5000 in your browser to watch runs appear")

Connected to MLflow at localhost:5000
Open localhost:5000 in your browser to watch runs appear


## Train XGBoost — Run 1 (v1 — default hyperparameters)
First training run to establish a baseline ML performance.
If F1 improvement is below 15%, we tune and run again as a new MLflow run.

In [21]:
# ── Step 4: Train and log everything to MLflow ───────────────

with mlflow.start_run(run_name="xgboost-v1"):
    model = XGBClassifier(
        n_estimators=300,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=spw,
        eval_metric='logloss',
        use_label_encoder=False,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train,
              eval_set=[(X_test, y_test)], verbose=50)

    # ── Step 5: Measure performance on test data ──────────────
    # This MUST be the same X_test you used for the rule baseline.
    y_pred = model.predict(X_test)
    ml_f1  = f1_score(y_test, y_pred)
    ml_pre = precision_score(y_test, y_pred)
    ml_rec = recall_score(y_test, y_pred)
    tn,fp,fn,tp = confusion_matrix(y_test, y_pred).ravel()

    # ── Step 6: Log metrics to MLflow ────────────────────────
    mlflow.log_params({"n_estimators":300,"max_depth":8,"lr":0.05})
    mlflow.log_metrics({"f1":ml_f1,"precision":ml_pre,"recall":ml_rec})
    mlflow.sklearn.log_model(model,"xgboost-detector")

    
    # ── Step 7: Load baseline and compare immediately ─────────
    with open("models/baseline/baseline_metrics.json") as bf:
        baseline = json.load(bf)
    improvement = ml_f1 - baseline["f1"]

    print(f"\nBaseline F1: {baseline['f1']:.4f}")
    print(f"ML F1:       {ml_f1:.4f}")
    print(f"Improvement: +{improvement:.4f} ({improvement*100:.1f}%)")
    print(f"Target met:  {'YES ✓' if improvement>=0.15 else 'NO ✗ tune model'}")

[0]	validation_0-logloss:0.65714


C:\Users\OlusegunFajobi\Downloads\My_Projects\Machine-Learning\SecOpsAI\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [14:33:43] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[50]	validation_0-logloss:0.23335
[100]	validation_0-logloss:0.31324
[150]	validation_0-logloss:0.35605
[200]	validation_0-logloss:0.36541
[250]	validation_0-logloss:0.37202
[299]	validation_0-logloss:0.39151


C:\Users\OlusegunFajobi\Downloads\My_Projects\Machine-Learning\SecOpsAI\venv\Lib\site-packages\_distutils_hack\__init__.py:18: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
C:\Users\OlusegunFajobi\Downloads\My_Projects\Machine-Learning\SecOpsAI\venv\Lib\site-packages\_distutils_hack\__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")



Baseline F1: 0.0472
ML F1:       0.8673
Improvement: +0.8201 (82.0%)
Target met:  YES ✓


In [23]:
os.makedirs("models/ml", exist_ok=True)
joblib.dump(model, "models/ml/xgboost_detector.pkl")
ml_metrics = {"f1":ml_f1,"precision":ml_pre,"recall":ml_rec,
              "fpr":fp/(fp+tn),"tp":int(tp),"fp":int(fp),
              "fn":int(fn),"tn":int(tn)}
with open("models/ml/ml_metrics.json","w") as f:
    json.dump(ml_metrics, f, indent=2)
print("Model saved: models/ml/xgboost_detector.pkl")
print("Metrics saved: models/ml/ml_metrics.json")

Model saved: models/ml/xgboost_detector.pkl
Metrics saved: models/ml/ml_metrics.json
